[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/drdave-teaching/OPIM5509-notebooks/blob/main/Module4/RNNs_By_Hand_basic.ipynb)

# RNNs by Hand
--------------------------------
**Dr. Dave Wanik - University of Connecticut**
Being able to count the trainable parameters by hand and describing the output shape of each layer will help you ensure that you actually know how these algorithms work. It will crystallize why you need to prep your data as 3D tensors.

Here's a cheat sheat for counting parms in deep learning models:
* **Link:** https://towardsdatascience.com/counting-no-of-parameters-in-deep-learning-models-by-hand-8f1716241889

And here's the blog with animated RNN, LSTM and GRU
* **Link:** https://towardsdatascience.com/animated-rnn-lstm-and-gru-ef124d06cf45

In [1]:
from tensorflow.keras.layers import Input, Dense, SimpleRNN, LSTM, GRU, Conv2D
from tensorflow.keras.layers import Bidirectional
from tensorflow.keras.models import Model
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding

# reproducibility: same seed every run (numbers on CPU match exactly; a GPU may drift a little)
import keras
keras.utils.set_random_seed(5509)


# Dense Neural Networks
(or feed-forward neural networks, FFNN)

* i, input size
* h, size of hidden layer
* o, output size
For one hidden layer,

```
num_params
= connections between layers + biases in every layer
= (i×h + h×o) + (h+o)
```

The example on the webpage assumes you have an input, a hidden layer and an output. For our examples with RNNs, we will assume o=0, and just use i and h. See below.

### One Simple RNN
One simpleRNN layer followed by a dense layer.

* `g`, no. of FFNNs in a unit (RNN has 1, GRU has 3, LSTM has 4)
* `h`, size of hidden units
* `i`, dimension/size of input

Since every FFNN (DNN) has `h(h+i) + h` parameters, we have
num_params = `g × [h(h+i) + h]`

Recall that the SimpleRNN only has one 'gate' or FFNN (you can see this in the cell!)

![alt text](https://miro.medium.com/max/1928/1*xn5kA92_J5KLaKcP7BMRLA.gif)



🔴
<!-- 🎙 DAVE TALKING POINTS — M4 · 2 — The vanilla RNN, one time step at a time
- Three features meet two hidden units (the red dots): a 5-input dense net with tanh.
- The handoff: each time step's hidden state feeds the next, so the final state has seen the whole window.
- SimpleRNN(units) sets the red-dot count, and the output shape is ALWAYS units - independent of look-back.
- Data prep is the whole game: (samples, look-back, features) - the deck of cards.
-->


## One Simple RNN Layer (basic)

In [2]:
# here's the script for the image above

# for an SimpleRNN, the input shape is "input_shape=(n_steps, n_features)"
# this corresponds to the graph in "Animated!"
n_steps=50 # doesn't matter!
n_features=3 # the 3 green dots... APPL, GOOGLE, FB
model = Sequential()
# parms in SimpleRNN is
model.add((SimpleRNN(2, activation='relu', input_shape=(n_steps, n_features)))) # the two red dots

# it is those 2 red dots that will go into the dense layer (don't forget to add 1 for the bias!)
model.add(Dense(1)) # this dense layer is not show in the animation, but it's needed! # predict netflix!
model.summary()

C:\Users\dww05002\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn (SimpleRNN)          │ (None, 2)              │            12 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │             3 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 15 (60.00 B)

 Trainable params: 15 (60.00 B)

 Non-trainable params: 0 (0.00 B)

In [3]:
# try the math
# for each layer


# TRAINABLE PARAMETERS
# the general RNN layer formula is g × [h(h+i) + h]
g = 1 # there's only 1 FFNN in a simpleRNN cell (look above!)
h = 2
i = 3 # this is number of features, not the lookback!
print(g*(h*(h+i) + h))

# SHAPE
# (None, 2) where 2 are the number of hidden units
# so the time series is now just a flattened input of 2 going into a dense layer
# this is what the 2 in simpleRNN(2) means! just 2 red dots.


# dense layer
# TRAINABLE PARAMETERS
# the dense layer is (i×h + h×o) + (h+o)
# but we ignore h since there is not output
h = 1 # the dense layer has a 1
i = 2 # 2 hidden node inputs
o = 0 # there is no output
print((i*h + h*o) + (h+o))

# output shape is (NONE,1)

12
3


🔴
<!-- 🎙 DAVE TALKING POINTS — M4 · 3 — Trainable parameters and output shape of a SimpleRNN
- Cell = (features + units) x units + bias. H=2, I=3 -> 12; plus a 1-unit dense head (3) = 15.
- Bigger: 30 features, 25 units -> (30+25) x 25 + 25 = 1,400; plus the dense = 1,426.
- The general formula: G x [H(H+I) + H]. G = number of little networks in the cell; SimpleRNN G=1.
- Match every number to model.summary() on screen - Assignment 5 grades exactly this.
-->


## One Simple RNN Layer (advanced)

In [4]:
# here's a related quiz question

# for an SimpleRNN, the input shape is "input_shape=(n_steps, n_features)"
# this corresponds to the graph in "Animated!"
n_steps=50
n_features=30 # having 30 stocks for covariates
model = Sequential()
model.add((SimpleRNN(25, activation='relu', input_shape=(n_steps, n_features))))
# it is those 25 red dots going into the dense layer, so you need 26 parms
model.add(Dense(1))
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn_1 (SimpleRNN)        │ (None, 25)             │         1,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            26 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,426 (5.57 KB)

 Trainable params: 1,426 (5.57 KB)

 Non-trainable params: 0 (0.00 B)

In [5]:
# try the math
# for each layer


# TRAINABLE PARAMETERS
# the general RNN layer formula is g × [h(h+i) + h]
g = 1
h = 25
i = 30 # this is number of features, not the lookback!
print(g*(h*(h+i) + h)) #answer = 1400

# SHAPE
# (None, 25) where 25 are the number of hidden units
# so the time series is now just a flattened input of 25 going into a dense layer
# this is what the 25 in simpleRNN(25) means! just 25 red dots.


# dense layer
# TRAINABLE PARAMETERS
# the dense layer is (i×h + h×o) + (h+o)
# but we ignore h since there is not output
h = 1 # the dense layer has a 1
i = 25
o = 0 # there is no output
print((i*h + h*o) + (h+o)) #answer = 26

# output shape is (NONE,1)

1400
26


🔴
<!-- 🎙 DAVE TALKING POINTS — M4 · 4 — LSTM by hand: four networks and a cell state
- G=4: four networks, plus a CELL state (long memory) beside the HIDDEN state (recent memory) - that's what fixes the vanishing gradient.
- 5 inputs x 2 units + 2 bias = 12 per network, x4 = 48. Same formula, G=4.
- Why it's slow: every time step spins all four networks.
- Forget / input / output gates decide what to keep - but at heart it's four nets fit at once.
-->


## One LSTM Layer (basic)
Here is what an LSTM looks like. Recall that it has four 'gates' or FFNNs.

![alt text](https://miro.medium.com/max/2250/1*goJVQs-p9kgLODFNyhl9zA.gif)

In [6]:
# here is the code that corresponds to the image

# for an LSTM, the input shape is "input_shape=(n_steps, n_features)"
# same example as above, just presented a different way
n_steps= 50 # doesn't matter! it will loop.
n_features= 3 # these are the 3 green dots
model = Sequential()
model.add((LSTM(2,  # these are the 2 red dots
                activation='relu', input_shape=(n_steps, n_features))))
model.add(Dense(1)) # not shown, but you need it and should realize that the
                    # 2 dark red dots are what will go into the dense layer
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 2)              │            48 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │             3 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 51 (204.00 B)

 Trainable params: 51 (204.00 B)

 Non-trainable params: 0 (0.00 B)

In [7]:
# try the math

# simple RNN
# TRAINABLE PARAMETERS
# the generic RNN layer is g × [h(h+i) + h]
g = 4 #LSTM has 4 FFNNs!
h = 2 # hidden units within LSTM, the two red dots
i = 3 # this is number of features, not the lookback! these are your 3 stocks (green dots!)
print(g*(h*(h+i) + h))

# SHAPE
# (None, 2) where 2 are the number of hidden units
# so the time series is now just a flattened input of 2 going into a dense layer

# dense layer
# TRAINABLE PARAMETERS
# the dense layer is (i×h + h×o) + (h+o)
# but we ignore h since there is not output
h = 1 # the dense layer has 1 output
i = 2 # these are all 4 inputs going into a dense layer
o = 0 # there is no output
print((i*h + h*o) + (h+o))

48
3


## One LSTM Layer (advanced)

In [8]:
# for an LSTM, the input shape is "input_shape=(n_steps, n_features)"
# same example as above, just presented a different way
n_steps= 30 # lookback
n_features= 5 # 5 different stocks, 5 green dots
model = Sequential()
model.add((LSTM(4, activation='relu', input_shape=(n_steps, n_features)))) # hidden units = 4 means 4 red dots
model.add(Dense(1))
model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_1 (LSTM)                   │ (None, 4)              │           160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │             5 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 165 (660.00 B)

 Trainable params: 165 (660.00 B)

 Non-trainable params: 0 (0.00 B)

In [9]:
# try the math

# TRAINABLE PARAMETERS
# the generic RNN forumla is g × [h(h+i) + h]
g = 4 #LSTM has 4! these are the 4 FFNNs
h = 4 # hidden units within LSTM (you get to decide this! it's the red dots...)
i = 5 # this is number of features, not the lookback! the green dots... your 5 stocks
print(g*(h*(h+i) + h)) #answer = 160

# SHAPE
# (None, 4) where 4 are the number of hidden units
# so the time series is now just a flattened input of 4 going into a dense layer

# dense layer
# TRAINABLE PARAMETERS
# the dense layer is (i×h + h×o) + (h+o)
# but we ignore h since there is not output
h = 1 # the dense layer has a 1
i = 4 # these are all 4 inputs going into a dense layer
o = 0 # there is no output
print((i*h + h*o) + (h+o)) #answer = 5

160
5


🔴
<!-- 🎙 DAVE TALKING POINTS — M4 · 5 — GRU by hand, and stacking/mixing cells
- G=3 (reset and update gates), counted the same way x3; reset_after=False makes Keras match the hand math - say it so summary() doesn't contradict you.
- Then the appendix: stack cells, mix SimpleRNN -> LSTM -> GRU - one-word swaps in Keras, everything else identical.
- Keep it under 8 minutes: the 2022 LSTM+GRU video ran 10:06 - this is its second half.
-->


## One GRU Layer (basic)
This is what a GRU looks like - note that it has three 'gates'.

![alt text](https://miro.medium.com/max/2214/1*lNNJOWnMjxLzdUnUQqwKcw.gif)

Caution: TensorFlow version difference!
Link: https://stackoverflow.com/questions/57318930/calculating-the-number-of-parameters-of-a-gru-layer-keras

Be careful of the bias term! Otherwise you need to add

In [10]:
# here is the example from the image
# and here is a related example
n_steps=50 # doesn't matter
n_features=3 # three stocks (FB, APPL, GOOG), three green dots
model = Sequential()
model.add((GRU(2, activation='relu', input_shape=(n_steps, n_features), # 2 red dots
               reset_after=False)))  # try this as False - helps math work out
model.add(Dense(1))
model.summary()

# if you don't say reset_after = False, you should add the bias terms
# which are bias_shape = (2, 3 * self.units)

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru (GRU)                       │ (None, 2)              │            36 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 1)              │             3 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 39 (156.00 B)

 Trainable params: 39 (156.00 B)

 Non-trainable params: 0 (0.00 B)

In [11]:
# here is the math for that example
# try the math

# gru
# TRAINABLE PARAMETERS
# the general RNN layer is g × [h(h+i) + h]
g = 3 #GRU has 3 FFNNs (this is ALWAYS TRUE for GRU)
h = 2 # hidden units within GRU (RED DOTS)
i = 3 # this is number of features, not the lookback! (GREEN DOTS)
print('# of trainable parms in gru_1 = ', g*(h*(h+i) + h))

# SHAPE
# (None, 2) where 2 are the number of hidden units
# so the time series is now just a flattened input of 2 going into a dense layer

# dense layer
# TRAINABLE PARAMETERS
# the dense layer is (i×h + h×o) + (h+o)
# but we ignore h since there is not output
h = 1 # the dense layer has a 1
i = 2 # these are all 2 inputs going into a dense layer
o = 0 # there is no output
print((i*h + h*o) + (h+o))

# of trainable parms in gru_1 =  36
3


## One GRU Layer (advanced)

In [12]:
# and here is a related example
n_steps=30000000 # so many time steps!
n_features=5 # five stocks = five green dots = FB, GOOG, APPL, GE, AMD
model = Sequential()
model.add((GRU(4, activation='relu', input_shape=(n_steps, n_features), # 4 red dots
               reset_after=False)))  # try this as False for no extra bias
model.add(Dense(1))
model.summary()

# if you don't say reset_after = False, you should add the bias terms
# which are bias_shape = (2, 3 * self.units)

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru_1 (GRU)                     │ (None, 4)              │           120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │             5 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 125 (500.00 B)

 Trainable params: 125 (500.00 B)

 Non-trainable params: 0 (0.00 B)

In [13]:
# try the math

# gru
# TRAINABLE PARAMETERS
# the general RNN layer is g × [h(h+i) + h]
g = 3 #GRU has 3!
h = 4 # hidden units within GRU
i = 5 # this is number of features, not the lookback!
print('# of trainable parms in gru_1 = ', g*(h*(h+i) + h))
print(g*(h*(h+i) + h))

# SHAPE
# (None, 4) where 4 are the number of hidden units
# so the time series is now just a flattened input of 4 going into a dense layer

# dense layer
# TRAINABLE PARAMETERS
# the dense layer is (i×h + h×o) + (h+o)
# but we ignore h since there is not output
h = 1 # the dense layer has a 1
i = 4 # these are all 4 inputs going into a dense layer
o = 0 # there is no output
print((i*h + h*o) + (h+o))

# of trainable parms in gru_1 =  120
120
5


# Advanced (stacking, mixing and matching.)
We will cover this in future lectures - provided as FYI.


### Two GRU layers going into a SimpleRNN
This is the fun part! Since you are returning sequences - the output shape will be 3D... you are storing all outputs from the DNNs within each GRU layer!

This is where n_steps actually gets used in the output size. Don't forget to set `return_sequences=True` when stacking layers - except for the last one that goes into the Dense layer.

In [14]:
n_steps=30 # this matters for output shape when we return sequences!
n_features=5 # these are 5 stocks (FB, APPL, GE, NETFLIX, AMD)

model = Sequential()
model.add((GRU(4, return_sequences=True, activation='relu', input_shape=(n_steps, n_features))))
model.add((GRU(2, return_sequences=True, activation='relu')))
model.add((SimpleRNN(25, activation='relu')))
model.add(Dense(1))
model.summary()

Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru_2 (GRU)                     │ (None, 30, 4)          │           132 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_3 (GRU)                     │ (None, 30, 2)          │            48 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_2 (SimpleRNN)        │ (None, 25)             │           700 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 1)              │            26 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 906 (3.54 KB)

 Trainable params: 906 (3.54 KB)

 Non-trainable params: 0 (0.00 B)

In [15]:
# try the math

# gru_1
# TRAINABLE PARAMETERS
# the general RNN layer formula is g × [h(h+i) + h]
g = 3 #GRU has 3!
h = 4 # hidden units within GRU
i = 5 # this is number of features, not the lookback!
print(g*(h*(h+i) + h)) #answer = 132

# SHAPE
# (None, 30, 4) where:
# 30 is the number of time steps (yes, it's appeared now!)
# 4 are the number of hidden units
# so the time series is now a derivative time series - it's a time series of
# dark red dots from the animation!

# gru_2
# TRAINABLE PARAMETERS
# the general RNN layer is g × [h(h+i) + h]
g = 3 #GRU has 3!
h = 2 # hidden units within GRU (we get to choose this)
i = 4 # this is number of features, not the lookback! this is inherited from previous layer
print(g*(h*(h+i) + h)) #answer = 132

# SHAPE
# (None, 4) where 4 are the number of hidden units
# so the time series is now just a flattened input of 4 going into a dense layer


# dense layer
# TRAINABLE PARAMETERS
# the dense layer is (i×h + h×o) + (h+o)
# but we ignore h since there is not output
h = 1 # the dense layer has a 1
i = 4 # these are all 4 inputs going into a dense layer
o = 0 # there is no output
print((i*h + h*o) + (h+o)) #answer = 5

120
42
5


### One SimpleRNN going into an LSTM
Left to students as an exercise.

In [16]:
n_steps=15 # lookback
n_features=30 # 30 different stocks

model = Sequential()
model.add((SimpleRNN(20, return_sequences=True, activation='relu', input_shape=(n_steps, n_features))))
model.add((LSTM(4, activation='relu'))) # see how there is NO RETURN SEQUENCES!!!
model.add(Dense(1))                             # you just keep the last hidden state
model.summary()

Model: "sequential_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn_3 (SimpleRNN)        │ (None, 15, 20)         │         1,020 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 4)              │           400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 1)              │             5 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,425 (5.57 KB)

 Trainable params: 1,425 (5.57 KB)

 Non-trainable params: 0 (0.00 B)

### Monster #1
Left as an exercise for students.

In [17]:
n_steps=30
n_features=30
model = Sequential()
model.add((SimpleRNN(30, return_sequences=True, activation='relu', input_shape=(n_steps, n_features))))
model.add((GRU(30, return_sequences=True,activation='relu')))
model.add((LSTM(30,activation='relu')))
model.add((Dense(30,activation='relu')))
model.add(Dense(1))
model.summary()

Model: "sequential_8"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn_4 (SimpleRNN)        │ (None, 30, 30)         │         1,830 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_4 (GRU)                     │ (None, 30, 30)         │         5,580 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 30)             │         7,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 30)             │           930 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 1)              │            31 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 15,691 (61.29 KB)

 Trainable params: 15,691 (61.29 KB)

 Non-trainable params: 0 (0.00 B)

### Monster #2
Left as an exercise for students.

In [18]:
n_steps=50
n_features=40
model = Sequential()
model.add((SimpleRNN(30, return_sequences=True, activation='relu', input_shape=(n_steps, n_features))))
model.add((GRU(20, return_sequences=True,activation='relu')))
model.add((GRU(25, return_sequences=True,activation='relu')))
model.add((GRU(22, return_sequences=True,activation='relu')))
model.add((GRU(21, return_sequences=True,activation='relu')))
model.add((SimpleRNN(10,activation='relu')))
model.add((Dense(50,activation='relu')))
model.add((Dense(50,activation='relu')))
model.add((Dense(50,activation='relu')))
model.add((Dense(50,activation='relu')))
model.add(Dense(1))
model.summary()

Model: "sequential_9"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn_5 (SimpleRNN)        │ (None, 50, 30)         │         2,130 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_5 (GRU)                     │ (None, 50, 20)         │         3,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_6 (GRU)                     │ (None, 50, 25)         │         3,525 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_7 (GRU)                     │ (None, 50, 22)         │         3,234 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_8 (GRU)                     │ (None, 50, 21)         │         2,835 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_6 (SimpleRNN)        │ (None, 10)             │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 50)             │           550 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 50)             │         2,550 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 50)             │         2,550 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 50)             │         2,550 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 1)              │            51 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,415 (91.46 KB)

 Trainable params: 23,415 (91.46 KB)

 Non-trainable params: 0 (0.00 B)